# p142 Linked List Cycle II 解析笔记

- 题号：p142
- 题目英文名：Linked List Cycle II
- 题目中文名：环形链表 II
- 题目链接：https://leetcode.cn/problems/linked-list-cycle-ii/
- 题型：链表 / 快慢指针
- 难度：Medium
- 推荐优先级：高
- Java 原代码位置：`solutions-java/leetcode/p142_linked_list_cycle_ii/LinkedListCycleII.java`


## 1. 题目一句话总结

这道题要求我们不只是判断链表有没有环，还要进一步找出第一次进入环的那个节点。

本质上考的是 Floyd 快慢指针的两阶段应用：先判环，再利用第一次相遇的位置反推出入环点。


## 2. 题目理解与约束分析

### 2.1 题目要求

给一条单链表，如果链表中存在环，就返回入环的第一个节点；如果不存在环，就返回 `None`。

注意返回的不是布尔值，而是具体节点对象。

### 2.2 输入与输出

- 输入：链表头节点 `head`
- 输出：入环节点或 `None`
- 返回结果含义：若有环，返回首次进入环的位置；若无环，返回空

### 2.3 关键约束

- 链表可能为空
- 链表可能非常短
- 不能修改链表结构
- 更优做法应为 `O(1)` 额外空间

### 2.4 示例分析

输入：`[3,2,0,-4]`，尾节点连回下标 `1` 的节点，也就是值为 `2` 的节点。

链表形态相当于：

```text
3 -> 2 -> 0 -> -4
     ^         |
     |_________|
```

所以返回的应该是值为 `2` 的那个节点。


## 3. Java 原代码

```java
package leetcode.p142_linked_list_cycle_ii;

public class LinkedListCycleII {

    public static class ListNode {
        public int val;
        public ListNode next;
    }

    public static ListNode detectCycle(ListNode head) {
        if (head == null || head.next == null || head.next.next == null) {
            return null;
        }

        ListNode slow = head.next;
        ListNode fast = head.next.next;

        while (slow != fast) {
            if (fast.next == null || fast.next.next == null) {
                return null;
            }
            slow = slow.next;
            fast = fast.next.next;
        }

        fast = head;
        while (slow != fast) {
            slow = slow.next;
            fast = fast.next;
        }
        return slow;
    }
}
```


## 4. 先从 Java 原方案理解

这份 Java 原解法采用的是 Floyd 快慢指针，而且它的初始化方式很有特点：

- `slow = head.next`
- `fast = head.next.next`

也就是说它不是把两个指针都从 `head` 出发，而是先错开一步和两步。这样做的直接原因是：配合 `while (slow != fast)`，可以避免一开始就因为都指向 `head` 而误判“已经相遇”。

整个 Java 方案分成两个阶段：

1. 第一阶段判断是否有环
2. 第二阶段定位入环点

如果第一阶段里 `fast` 或 `fast.next` 走到空，说明链表无环；如果 `slow` 和 `fast` 在环内相遇，就进入第二阶段。


## 5. 从朴素思路到优化思路

### 5.1 最直接的想法

一个很直观的办法是用哈希集合：

1. 遍历链表
2. 把访问过的节点放进集合
3. 第一次遇到重复节点时，它就是入环点

### 5.2 为什么还要优化

- 这个方法需要 `O(n)` 额外空间
- 题目更经典、更优雅的做法是 `O(1)` 额外空间

### 5.3 优化方向

既然环形结构里快指针一定会追上慢指针，那么我们可以先利用这个性质判环；一旦相遇，再借助距离关系推出入环点。

这就是 Java 原方案采用的 Floyd 两阶段方法。


## 6. 核心算法讲解

### 6.1 算法名称

- Floyd 判环
- 快慢指针

### 6.2 为什么想到这个算法

因为在有环链表中，快指针每次走 2 步，慢指针每次走 1 步，只要一直走，快指针一定会在环内追上慢指针。

一旦有了第一次相遇点，我们就能通过距离关系把问题从“有没有环”推进到“入环点在哪里”。

### 6.3 关键状态

- `slow`：慢指针，每次走一步
- `fast`：快指针，第一阶段每次走两步
- `head`：链表头节点，第二阶段中作为重新出发的起点

### 6.4 步骤拆解

1. 先处理长度过短的链表
2. 按 Java 原解初始化 `slow = head.next`、`fast = head.next.next`
3. 第一阶段让快慢指针在链表中追赶
4. 若快指针到空，返回 `None`
5. 若快慢指针相遇，说明有环
6. 把 `fast` 重置回 `head`
7. 第二阶段让 `slow` 和 `fast` 都每次走一步
8. 再次相遇的位置就是入环点


## 7. 为什么第二次相遇就是入环点

这一段直接对应 Java 注释中的证明思路。

设：

- 从头节点到入环点距离为 `a`
- 从入环点到第一次相遇点距离为 `b`
- 从第一次相遇点再走到入环点距离为 `c`
- 环长为 `b + c`

第一次相遇时：

- 慢指针走了 `a + b`
- 快指针走了 `a + b + k * (b + c)`

又因为快指针速度是慢指针的 2 倍，所以：

```text
2 * (a + b) = a + b + k * (b + c)
=> a + b = k * (b + c)
=> a = c + (k - 1) * (b + c)
```

这说明：

- 从 `head` 到入环点，距离是 `a`
- 从第一次相遇点再往前走到入环点，距离等价于先走 `c`，再多绕若干整圈

所以一个指针从 `head` 出发，另一个从第一次相遇点出发，并且都每次走一步，它们一定会在入环点相遇。


In [ ]:
from __future__ import annotations

from dataclasses import dataclass


@dataclass(eq=False)
class ListNode:
    val: int
    next: ListNode | None = None


In [ ]:
class Solution:
    def detectCycle(self, head: ListNode | None) -> ListNode | None:
        # 这里刻意保留 Java 原代码的初始化方式，而不是改写成别的常见模板。
        if head is None or head.next is None or head.next.next is None:
            return None

        slow = head.next
        fast = head.next.next

        while slow is not fast:
            if fast.next is None or fast.next.next is None:
                return None
            slow = slow.next
            fast = fast.next.next

        fast = head
        while slow is not fast:
            slow = slow.next
            fast = fast.next

        return slow


## 8. 代码逐段讲解

### 8.1 初始化部分

这份代码先排除了长度小于 3 的链表，因为按照 Java 原解的初始化方式，需要安全访问 `head.next.next`。

### 8.2 第一阶段

第一阶段只负责判环：

- `slow` 每次走一步
- `fast` 每次走两步

如果 `fast` 无法继续走两步，就说明无环；如果两者相遇，说明有环。

### 8.3 第二阶段

相遇后，把 `fast` 放回头节点，然后让 `slow` 和 `fast` 每次都走一步。第二次相遇点就是入环点。

### 8.4 返回结果

如果无环，返回 `None`；如果有环，返回再次相遇时所在的那个节点对象。


## 9. 复杂度分析

- 时间复杂度：`O(n)`
- 为什么是这个时间复杂度：两个阶段都只是在链表上线性推进，没有嵌套遍历
- 空间复杂度：`O(1)`
- 为什么是这个空间复杂度：只用了常数个指针变量，没有额外集合或数组


## 10. 易错点

1. 只写出判环逻辑，却忘了题目要求返回入环节点而不是布尔值。
2. 第一次相遇后直接返回 `slow`，这通常只是相遇点，不一定是入环点。
3. 按 Java 初始化方式写代码时，没有先判断 `head.next.next` 是否存在。
4. 第二阶段忘了把其中一个指针放回 `head`。
5. 把第二阶段也写成快指针两步、慢指针一步，结果就找不到入环点了。


## 11. 面试时怎么讲

可以这样概括：

> 这题我会用 Floyd 快慢指针做，两阶段完成。
> 第一阶段让快指针每次走两步、慢指针每次走一步，用来判断是否有环。
> 如果相遇，说明有环；然后把一个指针放回头节点，让两个指针都改成每次走一步。
> 根据距离关系，它们第二次相遇的位置就是入环点。
> 整体时间复杂度 `O(n)`，额外空间复杂度 `O(1)`。


## 12. 实际应用场景

这题可以类比成：在一个可能存在循环引用的单向引用链里，既要识别循环，又要找出第一次进入循环的位置。

### 具体业务案例：工作流错误回跳定位

#### 业务背景

某些工作流系统中，如果配置错误，流程节点可能会形成回跳，最终出现死循环。

#### 输入是什么

输入是一条按 `next` 串联的流程节点链。

#### 算法介入点

系统要判断这条流程链有没有循环，并且想知道是从哪个节点开始进入循环的。

#### 输出是什么

输出是导致循环开始的那个节点，方便定位配置问题。

#### 价值是什么

它解决的不只是“是否死循环”，而是“死循环从哪里开始”，这对排查问题更有价值。


In [ ]:
def build_cycle_list(values: list[int], pos: int) -> ListNode | None:
    if not values:
        return None

    nodes = [ListNode(v) for v in values]
    for i in range(len(nodes) - 1):
        nodes[i].next = nodes[i + 1]

    if pos != -1:
        nodes[-1].next = nodes[pos]

    return nodes[0]


head1 = build_cycle_list([3, 2, 0, -4], 1)
entry1 = Solution().detectCycle(head1)
print('示例一入环点:', entry1.val if entry1 else None)

head2 = build_cycle_list([1, 2], 0)
entry2 = Solution().detectCycle(head2)
print('示例二入环点:', entry2.val if entry2 else None)

head3 = build_cycle_list([1], -1)
entry3 = Solution().detectCycle(head3)
print('无环情况:', entry3)


## 13. Demo 输出说明

- 第一组应输出 `2`，说明能正确定位经典示例的入环点。
- 第二组应输出 `1`，说明即使环从头节点开始也能找到。
- 第三组应输出 `None`，说明无环短链表时会安全返回空。


## 14. 一句话复盘

> 这题最重要的不是记住快慢指针能判环，而是理解为什么第一次相遇后再同步前进，第二次相遇就一定是入环点。
